# 03. Huấn luyện Mô hình & Tinh chỉnh Siêu tham số (Modeling & Tuning)
**Đề tài:** Dự đoán khả năng khách hàng hủy đặt phòng khách sạn (*Hotel Booking Demand*)  
**Dữ liệu đầu vào:** `train_strat.csv`, `train_oot.csv` và `feature_metadata.json` từ Notebook 02 (tuyệt đối không sử dụng `test_strat.csv` và `test_oot.csv` trong notebook này).

## Kiến trúc thực nghiệm (KISS)
1. **Tiền xử lý chống rò rỉ trong từng Training Fold (`ColumnTransformer`):**
   - Biến số: `SimpleImputer(strategy='median')` (+ `StandardScaler()` cho `Logistic Regression`).
   - Biến phân loại ít giá trị: `OneHotEncoder(handle_unknown='ignore')`.
   - Biến phân loại nhiều giá trị (`country`, `agent`, `company`): `OneHotEncoder(handle_unknown='ignore', min_frequency=50)` để giảm số chiều an toàn và tránh vấn đề cross-fitting của `TargetEncoder`.
2. **Cross-Validation chống trùng mẫu (`StratifiedGroupKFold`):** 5 folds nhóm theo `fingerprint` (tính trên đúng danh sách `feature_cols` như Notebook 02, không chứa target hay biến leakage).
3. **Stratified Experiment (`train_strat.csv`):** Benchmark 6 mô hình cơ sở $
ightarrow$ chọn Top 2 theo CV ROC-AUC để tinh chỉnh bằng **Optuna (`20 trials`/mô hình)** $
ightarrow$ chọn mô hình tốt nhất tổng thể dựa trên CV ROC-AUC và lưu `best_model_pipeline.joblib`.
4. **Independent OOT Experiment (`train_oot.csv`):** Thực hiện thí nghiệm hoàn toàn độc lập (dữ liệu, fingerprint, CV và preprocessor riêng) trên 3 thuật toán đại diện (`Logistic Regression`, `Random Forest`, `LightGBM`) $
ightarrow$ tinh chỉnh Top 1 OOT (`15 trials`) và lưu `best_oot_model_pipeline.joblib`.

> **Lưu ý phương pháp:** Điểm **CV Score** trong notebook này chỉ dùng cho *Model Selection* và *Hyperparameter Tuning*; đánh giá cuối cùng (*Final Evaluation*) trên `test_strat.csv` và `test_oot.csv` được tách riêng sang Notebook 04.

In [1]:
from pathlib import Path
import json
import pickle
import time
import joblib
import numpy as np
import pandas as pd
import optuna

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
HIGH_CARD_MIN_FREQ = 50
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with (PROCESSED_DIR / "feature_metadata.json").open("r", encoding="utf-8") as f:
    meta = json.load(f)

target_col = meta["target_col"]
num_cols = meta["num_cols"]
low_card_cols = meta["low_card_cat_cols"]
high_card_cols = meta["high_card_cat_cols"]
feature_cols = num_cols + low_card_cols + high_card_cols
cat_dtypes = {c: str for c in low_card_cols + high_card_cols}

train_strat_df = pd.read_csv(PROCESSED_DIR / "train_strat.csv", dtype=cat_dtypes)
X_train_strat = train_strat_df[feature_cols].copy()
y_train_strat = train_strat_df[target_col].astype(int).values

print(f"Tập huấn luyện Stratified (train_strat): {X_train_strat.shape[0]:,} dòng x {X_train_strat.shape[1]} features")
print(f"Tỷ lệ hủy (train_strat): {y_train_strat.mean() * 100:.2f}%")
print(f"Nhóm đặc trưng: {len(num_cols)} số | {len(low_card_cols)} low-card | {len(high_card_cols)} high-card")

q:\Anaconda\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tập huấn luyện Stratified (train_strat): 95,367 dòng x 35 features
Tỷ lệ hủy (train_strat): 37.08%
Nhóm đặc trưng: 25 số | 7 low-card | 3 high-card


## 1. Hàm Pipeline Tiền xử lý & 5-Fold `StratifiedGroupKFold`
Mọi phép biến đổi học tham số (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`) chỉ được `fit` trên phần **Training Fold** rồi `transform` sang **Validation Fold**. Việc tính trước ma trận `float32` theo từng fold giúp Optuna chạy nhanh mà vẫn giữ nguyên tính tương đương toán học và không rò rỉ dữ liệu.

In [2]:
def build_preprocessor(scale_num: bool = False) -> ColumnTransformer:
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_num:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_steps), num_cols),
            ("low_cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_card_cols),
            (
                "high_cat",
                OneHotEncoder(handle_unknown="ignore", min_frequency=HIGH_CARD_MIN_FREQ, sparse_output=False),
                high_card_cols,
            ),
        ],
        remainder="drop",
    )


def compute_metrics(y_true, y_proba, threshold: float = 0.50) -> dict:
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "log_loss": log_loss(y_true, np.clip(y_proba, 1e-7, 1 - 1e-7)),
    }


def prepare_cv_folds(X_df: pd.DataFrame, y_arr: np.ndarray, n_splits: int = 5):
    # Fingerprint tính trên đúng feature_cols (nhất quán Notebook 02, không chứa target)
    groups = pd.util.hash_pandas_object(X_df[feature_cols], index=False)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    splits = list(sgkf.split(X_df, y_arr, groups=groups))

    unscaled_folds, scaled_folds = [], []
    max_overlap = 0
    for tr_idx, va_idx in splits:
        overlap = len(set(groups.iloc[tr_idx]).intersection(set(groups.iloc[va_idx])))
        max_overlap = max(max_overlap, overlap)

        X_tr_raw, X_va_raw = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

        prep_u = build_preprocessor(scale_num=False)
        X_tr_u = prep_u.fit_transform(X_tr_raw).astype(np.float32)
        X_va_u = prep_u.transform(X_va_raw).astype(np.float32)
        unscaled_folds.append((tr_idx, va_idx, X_tr_u, y_tr, X_va_u, y_va))

        prep_s = build_preprocessor(scale_num=True)
        X_tr_s = prep_s.fit_transform(X_tr_raw).astype(np.float32)
        X_va_s = prep_s.transform(X_va_raw).astype(np.float32)
        scaled_folds.append((tr_idx, va_idx, X_tr_s, y_tr, X_va_s, y_va))

    assert max_overlap == 0, f"Phát hiện {max_overlap} fingerprint chồng lấn giữa train/val fold!"
    return unscaled_folds, scaled_folds


def evaluate_cv(model_name: str, estimator, folds, n_samples: int):
    oof_proba = np.zeros(n_samples, dtype=np.float64)
    fold_metrics = []
    start = time.time()

    for _, va_idx, X_tr, y_tr, X_va, y_va in folds:
        clf = clone(estimator)
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_va)[:, 1]
        oof_proba[va_idx] = proba
        fold_metrics.append(compute_metrics(y_va, proba))

    elapsed = time.time() - start
    df_fm = pd.DataFrame(fold_metrics)
    summary = {
        "model": model_name,
        "roc_auc_mean": df_fm["roc_auc"].mean(),
        "roc_auc_std": df_fm["roc_auc"].std(),
        "pr_auc_mean": df_fm["pr_auc"].mean(),
        "f1_mean": df_fm["f1"].mean(),
        "precision_mean": df_fm["precision"].mean(),
        "recall_mean": df_fm["recall"].mean(),
        "accuracy_mean": df_fm["accuracy"].mean(),
        "log_loss_mean": df_fm["log_loss"].mean(),
        "fit_time_sec": round(elapsed, 2),
    }
    return summary, oof_proba


t0 = time.time()
strat_folds_u, strat_folds_s = prepare_cv_folds(X_train_strat, y_train_strat, n_splits=5)
print(
    f"Hoàn tất chuẩn bị 5-Fold StratifiedGroupKFold (Stratified) trong {time.time() - t0:.2f}s | "
    f"Số chiều sau OHE (fold 1): {strat_folds_u[0][2].shape[1]} | Fingerprint overlap: 0"
)

Hoàn tất chuẩn bị 5-Fold StratifiedGroupKFold (Stratified) trong 13.65s | Số chiều sau OHE (fold 1): 220 | Fingerprint overlap: 0


## 2. Stratified Experiment — Baseline Benchmark (6 Mô hình)
So sánh 6 mô hình trên cùng bộ 5-Fold `StratifiedGroupKFold` (đặt tên rõ *Baseline* / *Initial Config* thay vì *Default*). Mọi ứng viên đều được lưu trong từ điển `strat_candidates` để đảm bảo mô hình nào thắng trên CV cũng đều có thể được chọn và huấn luyện lại hợp lệ.

In [3]:
strat_initial_specs = [
    ("Logistic Regression (Baseline)", "logreg", LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE), True),
    ("Decision Tree (Baseline)", "dt", DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=RANDOM_STATE), False),
    ("Random Forest (Baseline)", "rf", RandomForestClassifier(n_estimators=250, max_depth=18, min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_STATE), False),
    ("XGBoost (Initial Config)", "xgb", XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE), False),
    ("LightGBM (Initial Config)", "lgbm", LGBMClassifier(n_estimators=300, learning_rate=0.08, num_leaves=63, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE), False),
    ("CatBoost (Initial Config)", "catboost", CatBoostClassifier(iterations=600, learning_rate=0.08, depth=6, eval_metric="Logloss", verbose=0, thread_count=-1, random_seed=RANDOM_STATE), False),
]

strat_cv_results = []
strat_oof_store = {}
strat_candidates = {}

for name, family, est, use_scaled in strat_initial_specs:
    folds = strat_folds_s if use_scaled else strat_folds_u
    res, oof = evaluate_cv(name, est, folds, len(y_train_strat))
    strat_cv_results.append(res)
    strat_oof_store[name] = oof
    strat_candidates[name] = {
        "family": family,
        "estimator": est,
        "use_scaled": use_scaled,
        "roc_auc_mean": res["roc_auc_mean"],
    }
    print(
        f"[{name:30s}] ROC-AUC: {res['roc_auc_mean']:.4f} ± {res['roc_auc_std']:.4f} | "
        f"PR-AUC: {res['pr_auc_mean']:.4f} | F1: {res['f1_mean']:.4f} | Time: {res['fit_time_sec']}s"
    )

df_strat_base = pd.DataFrame(strat_cv_results).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
print("\n=== Bảng Baseline Benchmark — Stratified 5-Fold StratifiedGroupKFold ===")
display(df_strat_base.round(4))

[Logistic Regression (Baseline)] ROC-AUC: 0.9016 ± 0.0056 | PR-AUC: 0.8624 | F1: 0.7375 | Time: 6.51s
[Decision Tree (Baseline)      ] ROC-AUC: 0.9199 ± 0.0040 | PR-AUC: 0.8768 | F1: 0.7768 | Time: 6.55s
[Random Forest (Baseline)      ] ROC-AUC: 0.9321 ± 0.0034 | PR-AUC: 0.9038 | F1: 0.7627 | Time: 22.96s
[XGBoost (Initial Config)      ] ROC-AUC: 0.9403 ± 0.0029 | PR-AUC: 0.9124 | F1: 0.8077 | Time: 10.25s
[LightGBM (Initial Config)     ] ROC-AUC: 0.9438 ± 0.0023 | PR-AUC: 0.9168 | F1: 0.8126 | Time: 7.26s
[CatBoost (Initial Config)     ] ROC-AUC: 0.9424 ± 0.0031 | PR-AUC: 0.9163 | F1: 0.8124 | Time: 30.38s

=== Bảng Baseline Benchmark — Stratified 5-Fold StratifiedGroupKFold ===


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_mean,precision_mean,recall_mean,accuracy_mean,log_loss_mean,fit_time_sec
0,LightGBM (Initial Config),0.9438,0.0023,0.9168,0.8126,0.8381,0.7887,0.8651,0.2844,7.26
1,CatBoost (Initial Config),0.9424,0.0031,0.9163,0.8124,0.8412,0.7857,0.8655,0.2890,30.38
2,XGBoost (Initial Config),0.9403,0.0029,0.9124,0.8077,0.8381,0.7797,0.8624,0.2933,10.25
3,Random Forest (Baseline),0.9321,0.0034,0.9038,0.7627,0.8871,0.6691,0.8457,0.3488,22.96
4,Decision Tree (Baseline),0.9199,0.0040,0.8768,0.7768,0.8183,0.7394,0.8424,0.3367,6.55
5,Logistic Regression (Baseline),0.9016,0.0056,0.8624,0.7375,0.7987,0.6852,0.8191,0.3673,6.51


## 3. Stratified Experiment — Tinh chỉnh Siêu tham số (Optuna Top 2)
Tự động chọn **Top 2 mô hình** có CV ROC-AUC cao nhất từ bảng Baseline Benchmark để chạy Optuna (`n_trials = 20`/mô hình, `TPESampler(seed=42)` + `MedianPruner`), tối ưu trực tiếp 5-Fold `StratifiedGroupKFold` ROC-AUC trên `train_strat.csv`. Sau tinh chỉnh, đối chiếu trực tiếp mức cải thiện so với cấu hình ban đầu (không mặc định bản Tuned luôn tốt hơn).

In [4]:
def build_tuned_estimator(family: str, trial_params: dict):
    if family == "lgbm":
        params = {**trial_params, "subsample_freq": 1, "n_jobs": -1, "verbose": -1, "random_state": RANDOM_STATE}
        return LGBMClassifier(**params), params
    elif family == "xgb":
        params = {**trial_params, "tree_method": "hist", "eval_metric": "logloss", "n_jobs": -1, "random_state": RANDOM_STATE}
        return XGBClassifier(**params), params
    elif family == "catboost":
        params = {**trial_params, "eval_metric": "Logloss", "verbose": 0, "thread_count": -1, "random_seed": RANDOM_STATE}
        return CatBoostClassifier(**params), params
    elif family == "rf":
        params = {**trial_params, "n_jobs": -1, "random_state": RANDOM_STATE}
        return RandomForestClassifier(**params), params
    raise ValueError(f"Chưa hỗ trợ tuning cho nhóm mô hình: {family}")


def suggest_params(family: str, trial: optuna.Trial) -> dict:
    if family == "lgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 140),
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 15, 100),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    elif family == "xgb":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 300, 950),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 2.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    elif family == "catboost":
        return {
            "iterations": trial.suggest_int("iterations", 450, 1100),
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.14, log=True),
            "depth": trial.suggest_int("depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 12.0),
            "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.5),
        }
    elif family == "rf":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 450),
            "max_depth": trial.suggest_int("max_depth", 10, 28),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 12),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
        }
    raise ValueError(f"Chưa hỗ trợ không gian tìm kiếm cho: {family}")


def tune_model_optuna(family: str, folds, n_trials: int = 20, timeout: int = 240):
    def objective(trial):
        trial_params = suggest_params(family, trial)
        clf_template, _ = build_tuned_estimator(family, trial_params)
        aucs = []
        for step, (_, _, X_tr, y_tr, X_va, y_va) in enumerate(folds):
            clf = clone(clf_template)
            clf.fit(X_tr, y_tr)
            proba = clf.predict_proba(X_va)[:, 1]
            aucs.append(roc_auc_score(y_va, proba))
            trial.report(float(np.mean(aucs)), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(aucs))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=n_trials, timeout=timeout)
    best_est, full_params = build_tuned_estimator(family, study.best_params)
    return study, best_est, full_params


# Tự động chọn Top 2 mô hình từ Baseline Benchmark (thuộc nhóm hỗ trợ tuning)
family_display_prefix = {"lgbm": "LightGBM", "xgb": "XGBoost", "catboost": "CatBoost", "rf": "Random Forest"}
top2_rows = [
    row for _, row in df_strat_base.iterrows()
    if strat_candidates[row["model"]]["family"] in family_display_prefix
][:2]

strat_tuned_params = {}
tuning_gains = []

for row in top2_rows:
    base_name = row["model"]
    base_auc = float(row["roc_auc_mean"])
    family = strat_candidates[base_name]["family"]
    tuned_name = f"{family_display_prefix[family]} (Optuna)"

    t0 = time.time()
    study, best_est, full_params = tune_model_optuna(family, strat_folds_u, n_trials=20, timeout=240)
    res_t, oof_t = evaluate_cv(tuned_name, best_est, strat_folds_u, len(y_train_strat))

    strat_cv_results.append(res_t)
    strat_oof_store[tuned_name] = oof_t
    strat_tuned_params[tuned_name] = full_params
    strat_candidates[tuned_name] = {
        "family": family,
        "estimator": best_est,
        "use_scaled": False,
        "roc_auc_mean": res_t["roc_auc_mean"],
    }
    delta = res_t["roc_auc_mean"] - base_auc
    tuning_gains.append({
        "Mô hình": family_display_prefix[family],
        "Initial CV ROC-AUC": round(base_auc, 5),
        "Tuned CV ROC-AUC": round(res_t["roc_auc_mean"], 5),
        "Chênh lệch (Δ)": round(delta, 5),
        "Số trials": len(study.trials),
        "Thời gian tune (s)": round(time.time() - t0, 1),
    })
    print(f"[{tuned_name:22s}] CV ROC-AUC: {res_t['roc_auc_mean']:.5f} (Δ = {delta:+.5f} so với {base_name}) | Trials: {len(study.trials)}")

print("\n=== Đánh giá mức cải thiện sau Optuna Tuning (Stratified) ===")
display(pd.DataFrame(tuning_gains))

[LightGBM (Optuna)     ] CV ROC-AUC: 0.94539 (Δ = +0.00155 so với LightGBM (Initial Config)) | Trials: 16
[CatBoost (Optuna)     ] CV ROC-AUC: 0.94383 (Δ = +0.00142 so với CatBoost (Initial Config)) | Trials: 7

=== Đánh giá mức cải thiện sau Optuna Tuning (Stratified) ===


,Mô hình,Initial CV ROC-AUC,Tuned CV ROC-AUC,Chênh lệch (Δ),Số trials,Thời gian tune (s)
0,LightGBM,0.94385,0.94539,0.00155,16,265.3
1,CatBoost,0.94241,0.94383,0.00142,7,290.6


## 4. Stratified Experiment — Model Selection & Lưu Artifacts
- Tổng hợp toàn bộ ứng viên (Baseline / Initial Config + Optuna Tuned) để chọn mô hình tốt nhất theo **5-Fold `StratifiedGroupKFold` CV ROC-AUC**.
- Đóng gói `Pipeline(preprocessor -> model)` đầy đủ, `fit` trên toàn bộ `train_strat.csv` và lưu mô hình tốt nhất (`best_model_pipeline.joblib`) cùng các mô hình đại diện của từng họ thuật toán vào `models/` để Notebook 04 thực hiện đánh giá cuối cùng trên `test_strat.csv`.

In [5]:
df_strat_all = pd.DataFrame(strat_cv_results).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
print("=== BẢNG TỔNG HỢP 5-FOLD STRATIFIED GROUP CV (STRATIFIED EXPERIMENT) ===")
display(df_strat_all.round(4))

# Chọn mô hình tốt nhất tổng thể (bao phủ 100% mọi ứng viên trong bảng)
best_strat_name = df_strat_all.iloc[0]["model"]
best_strat_info = strat_candidates[best_strat_name]

# Chọn phiên bản tốt nhất của mỗi họ thuật toán đại diện để Notebook 04 so sánh trên test_strat
rep_families = [
    ("logreg", "baseline_logreg_pipeline.joblib"),
    ("rf", "random_forest_pipeline.joblib"),
    ("lgbm", "lgbm_pipeline.joblib"),
    ("xgb", "xgb_pipeline.joblib"),
    ("catboost", "catboost_pipeline.joblib"),
    ("dt", "decision_tree_pipeline.joblib")
]

strat_model_registry = []
fitted_pipelines = {}

for family, filename in rep_families:
    fam_candidates = [
        (name, info) for name, info in strat_candidates.items() if info["family"] == family
    ]
    fam_best_name, fam_best_info = max(fam_candidates, key=lambda x: x[1]["roc_auc_mean"])
    pipe = Pipeline([
        ("preprocessor", build_preprocessor(scale_num=fam_best_info["use_scaled"])),
        ("model", clone(fam_best_info["estimator"])),
    ])
    t0 = time.time()
    pipe.fit(X_train_strat, y_train_strat)
    joblib.dump(pipe, MODELS_DIR / filename)
    fitted_pipelines[fam_best_name] = pipe
    strat_model_registry.append({"model": fam_best_name, "filename": filename, "family": family})
    print(f"Đã huấn luyện & lưu {filename:32s} <- {fam_best_name} ({time.time() - t0:.1f}s)")

# Nếu mô hình tốt nhất tổng thể là Decision Tree hoặc chưa nằm trong fitted_pipelines thì fit bổ sung
if best_strat_name not in fitted_pipelines:
    best_pipe = Pipeline([
        ("preprocessor", build_preprocessor(scale_num=best_strat_info["use_scaled"])),
        ("model", clone(best_strat_info["estimator"])),
    ])
    best_pipe.fit(X_train_strat, y_train_strat)
    fitted_pipelines[best_strat_name] = best_pipe

best_strat_pipe = fitted_pipelines[best_strat_name]
joblib.dump(best_strat_pipe, MODELS_DIR / "best_model_pipeline.joblib")

# Lưu bảng CV, OOF predictions và metadata siêu tham số cho Stratified Experiment
df_strat_all.to_csv(MODELS_DIR / "cv_benchmark_results.csv", index=False)
with (MODELS_DIR / "oof_predictions.pkl").open("wb") as f:
    pickle.dump({"y_train": y_train_strat, "oof": strat_oof_store}, f)

strat_feature_names_out = list(best_strat_pipe.named_steps["preprocessor"].get_feature_names_out())
strat_meta = {
    "experiment": "Stratified (StratifiedGroupKFold 5-fold)",
    "best_overall_model": best_strat_name,
    "best_overall_cv_roc_auc": round(float(df_strat_all.iloc[0]["roc_auc_mean"]), 5),
    "high_card_min_frequency": HIGH_CARD_MIN_FREQ,
    "strat_model_registry": strat_model_registry,
    "tuned_hyperparameters": strat_tuned_params,
    "feature_names_out": strat_feature_names_out,
}
with (MODELS_DIR / "best_hyperparameters.json").open("w", encoding="utf-8") as f:
    json.dump(strat_meta, f, ensure_ascii=False, indent=2)

print(f"\n>>> MÔ HÌNH TỐT NHẤT (STRATIFIED CV): {best_strat_name} | CV ROC-AUC = {df_strat_all.iloc[0]['roc_auc_mean']:.5f}")

=== BẢNG TỔNG HỢP 5-FOLD STRATIFIED GROUP CV (STRATIFIED EXPERIMENT) ===


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_mean,precision_mean,recall_mean,accuracy_mean,log_loss_mean,fit_time_sec
0,LightGBM (Optuna),0.9454,0.0026,0.9196,0.8173,0.8427,0.7937,0.8685,0.2814,19.61
1,LightGBM (Initial Config),0.9438,0.0023,0.9168,0.8126,0.8381,0.7887,0.8651,0.2844,7.26
2,CatBoost (Optuna),0.9438,0.0034,0.9181,0.8153,0.8408,0.7914,0.8671,0.2843,46.04
3,CatBoost (Initial Config),0.9424,0.0031,0.9163,0.8124,0.8412,0.7857,0.8655,0.2890,30.38
4,XGBoost (Initial Config),0.9403,0.0029,0.9124,0.8077,0.8381,0.7797,0.8624,0.2933,10.25
5,Random Forest (Baseline),0.9321,0.0034,0.9038,0.7627,0.8871,0.6691,0.8457,0.3488,22.96
6,Decision Tree (Baseline),0.9199,0.0040,0.8768,0.7768,0.8183,0.7394,0.8424,0.3367,6.55
7,Logistic Regression (Baseline),0.9016,0.0056,0.8624,0.7375,0.7987,0.6852,0.8191,0.3673,6.51


Đã huấn luyện & lưu baseline_logreg_pipeline.joblib  <- Logistic Regression (Baseline) (3.3s)
Đã huấn luyện & lưu random_forest_pipeline.joblib    <- Random Forest (Baseline) (6.2s)
Đã huấn luyện & lưu lgbm_pipeline.joblib             <- LightGBM (Optuna) (4.6s)
Đã huấn luyện & lưu xgb_pipeline.joblib              <- XGBoost (Initial Config) (2.4s)
Đã huấn luyện & lưu catboost_pipeline.joblib         <- CatBoost (Optuna) (8.4s)
Đã huấn luyện & lưu decision_tree_pipeline.joblib    <- Decision Tree (Baseline) (1.8s)

>>> MÔ HÌNH TỐT NHẤT (STRATIFIED CV): LightGBM (Optuna) | CV ROC-AUC = 0.94539


## 5. Independent OOT Experiment (Huấn luyện Độc lập trên `train_oot.csv`)
Để phép đánh giá theo thời gian ở Notebook 04 thực sự độc lập và không bị rò rỉ lựa chọn mô hình từ Stratified Experiment:
- Chỉ đọc `train_oot.csv` (`< 2017-05-01`), **tuyệt đối không đọc `test_oot.csv`**.
- Tạo mới hoàn toàn `fingerprint`, 5 folds `StratifiedGroupKFold` và các `ColumnTransformer` riêng trên `train_oot.csv` (không dùng chung dữ liệu, preprocessor hay siêu tham số từ Stratified).
- Benchmark riêng 3 thuật toán đại diện xác định trước: `Logistic Regression`, `Random Forest`, `LightGBM`, sau đó chỉ tinh chỉnh **Top 1 OOT** bằng Optuna (`15 trials`) và lưu `best_oot_model_pipeline.joblib`.

In [6]:
train_oot_df = pd.read_csv(PROCESSED_DIR / "train_oot.csv", dtype=cat_dtypes)
X_train_oot = train_oot_df[feature_cols].copy()
y_train_oot = train_oot_df[target_col].astype(int).values

print(f"Tập huấn luyện OOT độc lập (train_oot): {X_train_oot.shape[0]:,} dòng x {X_train_oot.shape[1]} features")
print(f"Tỷ lệ hủy (train_oot): {y_train_oot.mean() * 100:.2f}%")

t0 = time.time()
oot_folds_u, oot_folds_s = prepare_cv_folds(X_train_oot, y_train_oot, n_splits=5)
print(
    f"Hoàn tất chuẩn bị 5-Fold StratifiedGroupKFold (OOT) trong {time.time() - t0:.2f}s | "
    f"Số chiều sau OHE (fold 1): {oot_folds_u[0][2].shape[1]} | Fingerprint overlap: 0"
)

oot_initial_specs = [
    ("Logistic Regression (OOT Baseline)", "logreg", LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE), True),
    ("Random Forest (OOT Baseline)", "rf", RandomForestClassifier(n_estimators=250, max_depth=18, min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_STATE), False),
    ("LightGBM (OOT Initial Config)", "lgbm", LGBMClassifier(n_estimators=300, learning_rate=0.08, num_leaves=63, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1, random_state=RANDOM_STATE), False),
]

oot_cv_results = []
oot_oof_store = {}
oot_candidates = {}

for name, family, est, use_scaled in oot_initial_specs:
    folds = oot_folds_s if use_scaled else oot_folds_u
    res, oof = evaluate_cv(name, est, folds, len(y_train_oot))
    oot_cv_results.append(res)
    oot_oof_store[name] = oof
    oot_candidates[name] = {
        "family": family,
        "estimator": est,
        "use_scaled": use_scaled,
        "roc_auc_mean": res["roc_auc_mean"],
    }
    print(
        f"[{name:34s}] ROC-AUC: {res['roc_auc_mean']:.4f} ± {res['roc_auc_std']:.4f} | "
        f"PR-AUC: {res['pr_auc_mean']:.4f} | F1: {res['f1_mean']:.4f} | Time: {res['fit_time_sec']}s"
    )

df_oot_base = pd.DataFrame(oot_cv_results).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
display(df_oot_base.round(4))

Tập huấn luyện OOT độc lập (train_oot): 97,032 dòng x 35 features
Tỷ lệ hủy (train_oot): 36.28%
Hoàn tất chuẩn bị 5-Fold StratifiedGroupKFold (OOT) trong 16.63s | Số chiều sau OHE (fold 1): 227 | Fingerprint overlap: 0
[Logistic Regression (OOT Baseline)] ROC-AUC: 0.9106 ± 0.0037 | PR-AUC: 0.8743 | F1: 0.7488 | Time: 9.3s
[Random Forest (OOT Baseline)      ] ROC-AUC: 0.9414 ± 0.0029 | PR-AUC: 0.9166 | F1: 0.7818 | Time: 21.51s
[LightGBM (OOT Initial Config)     ] ROC-AUC: 0.9520 ± 0.0037 | PR-AUC: 0.9285 | F1: 0.8271 | Time: 4.71s


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_mean,precision_mean,recall_mean,accuracy_mean,log_loss_mean,fit_time_sec
0,LightGBM (OOT Initial Config),0.9520,0.0037,0.9285,0.8271,0.8588,0.7977,0.8790,0.2613,4.71
1,Random Forest (OOT Baseline),0.9414,0.0029,0.9166,0.7818,0.9076,0.6867,0.8610,0.3240,21.51
2,Logistic Regression (OOT Baseline),0.9106,0.0037,0.8743,0.7488,0.8149,0.6927,0.8314,0.3473,9.30


In [7]:
# Chỉ tinh chỉnh Top 1 ứng viên OOT (thuộc nhóm hỗ trợ tuning) trên oot_folds độc lập
top1_oot_row = next(
    row for _, row in df_oot_base.iterrows()
    if oot_candidates[row["model"]]["family"] in family_display_prefix
)
oot_base_name = top1_oot_row["model"]
oot_base_auc = float(top1_oot_row["roc_auc_mean"])
oot_top_family = oot_candidates[oot_base_name]["family"]
oot_tuned_name = f"{family_display_prefix[oot_top_family]} (OOT Optuna)"

t0 = time.time()
study_oot, best_oot_tuned_est, oot_full_params = tune_model_optuna(
    oot_top_family, oot_folds_u, n_trials=15, timeout=180
)
res_oot_t, oof_oot_t = evaluate_cv(oot_tuned_name, best_oot_tuned_est, oot_folds_u, len(y_train_oot))

oot_cv_results.append(res_oot_t)
oot_oof_store[oot_tuned_name] = oof_oot_t
oot_candidates[oot_tuned_name] = {
    "family": oot_top_family,
    "estimator": best_oot_tuned_est,
    "use_scaled": False,
    "roc_auc_mean": res_oot_t["roc_auc_mean"],
}
oot_delta = res_oot_t["roc_auc_mean"] - oot_base_auc
print(
    f"[{oot_tuned_name:24s}] OOT CV ROC-AUC: {res_oot_t['roc_auc_mean']:.5f} "
    f"(Δ = {oot_delta:+.5f} so với {oot_base_name}) | Trials: {len(study_oot.trials)} | Time: {time.time() - t0:.1f}s"
)

df_oot_all = pd.DataFrame(oot_cv_results).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
print("\n=== BẢNG TỔNG HỢP 5-FOLD STRATIFIED GROUP CV (INDEPENDENT OOT EXPERIMENT) ===")
display(df_oot_all.round(4))

# Chọn mô hình OOT tốt nhất theo OOT CV ROC-AUC (không mặc định bản Tuned luôn thắng)
best_oot_name = df_oot_all.iloc[0]["model"]
best_oot_info = oot_candidates[best_oot_name]

best_oot_pipe = Pipeline([
    ("preprocessor", build_preprocessor(scale_num=best_oot_info["use_scaled"])),
    ("model", clone(best_oot_info["estimator"])),
])
best_oot_pipe.fit(X_train_oot, y_train_oot)
joblib.dump(best_oot_pipe, MODELS_DIR / "best_oot_model_pipeline.joblib")

df_oot_all.to_csv(MODELS_DIR / "oot_cv_benchmark_results.csv", index=False)
with (MODELS_DIR / "oot_oof_predictions.pkl").open("wb") as f:
    pickle.dump({"y_train": y_train_oot, "oof": oot_oof_store}, f)

oot_meta = {
    "experiment": "Independent OOT (train_oot.csv < 2017-05-01, StratifiedGroupKFold 5-fold)",
    "best_oot_model": best_oot_name,
    "best_oot_cv_roc_auc": round(float(df_oot_all.iloc[0]["roc_auc_mean"]), 5),
    "high_card_min_frequency": HIGH_CARD_MIN_FREQ,
    "tuned_hyperparameters": {oot_tuned_name: oot_full_params},
    "feature_names_out": list(best_oot_pipe.named_steps["preprocessor"].get_feature_names_out()),
}
with (MODELS_DIR / "oot_best_hyperparameters.json").open("w", encoding="utf-8") as f:
    json.dump(oot_meta, f, ensure_ascii=False, indent=2)

print(f"\n>>> MÔ HÌNH TỐT NHẤT (INDEPENDENT OOT CV): {best_oot_name} | OOT CV ROC-AUC = {df_oot_all.iloc[0]['roc_auc_mean']:.5f}")
print("Đã lưu đầy đủ artifacts OOT độc lập vào MachineLearning/models/!")

[LightGBM (OOT Optuna)   ] OOT CV ROC-AUC: 0.95269 (Δ = +0.00068 so với LightGBM (OOT Initial Config)) | Trials: 15 | Time: 154.6s

=== BẢNG TỔNG HỢP 5-FOLD STRATIFIED GROUP CV (INDEPENDENT OOT EXPERIMENT) ===


,model,roc_auc_mean,roc_auc_std,pr_auc_mean,f1_mean,precision_mean,recall_mean,accuracy_mean,log_loss_mean,fit_time_sec
0,LightGBM (OOT Optuna),0.9527,0.0036,0.9296,0.8278,0.8613,0.7969,0.8798,0.2596,14.31
1,LightGBM (OOT Initial Config),0.9520,0.0037,0.9285,0.8271,0.8588,0.7977,0.8790,0.2613,4.71
2,Random Forest (OOT Baseline),0.9414,0.0029,0.9166,0.7818,0.9076,0.6867,0.8610,0.3240,21.51
3,Logistic Regression (OOT Baseline),0.9106,0.0037,0.8743,0.7488,0.8149,0.6927,0.8314,0.3473,9.30



>>> MÔ HÌNH TỐT NHẤT (INDEPENDENT OOT CV): LightGBM (OOT Optuna) | OOT CV ROC-AUC = 0.95269
Đã lưu đầy đủ artifacts OOT độc lập vào MachineLearning/models/!


## 6. Tóm tắt & Ghi chú Đồng bộ cho Notebook 04 (`04_evaluation_report.ipynb`)
Toàn bộ mô hình và kết quả CV mới đã được xuất vào `MachineLearning/models/`. Ở đợt cập nhật Notebook 04 tiếp theo, cần đồng bộ các điểm sau:
1. **Đọc danh sách mô hình Stratified động từ `best_hyperparameters.json` (`strat_model_registry`):** Thay vì hard-code tên file cũ (`lgbm_tuned_pipeline.joblib`, `xgb_tuned_pipeline.joblib`, `catboost_tuned_pipeline.joblib`), đọc trực tiếp từ `strat_model_registry` (`baseline_logreg_pipeline.joblib`, `random_forest_pipeline.joblib`, `lgbm_pipeline.joblib`, `xgb_pipeline.joblib`, `catboost_pipeline.joblib`) và `best_model_pipeline.joblib`.
2. **Đánh giá OOT độc lập thực sự:** Tải trực tiếp `best_oot_model_pipeline.joblib`, `oot_cv_benchmark_results.csv` và `oot_oof_predictions.pkl` để tối ưu ngưỡng OOT riêng (nếu cần) và đánh giá trên `test_oot.csv` — **bỏ bước clone `best_model_pipeline.joblib` từ Stratified Experiment**.
3. **Cập nhật lại toàn bộ biểu đồ đánh giá (`07`–`12`):** Chạy lại Notebook 04 để làm mới `test_evaluation_results.csv`, `oot_validation_results.csv` và các biểu đồ tương ứng với dữ liệu và mô hình mới.